# Machine Learning 지도학습 - 분류


## ⚙️환경설정


In [1]:
from turtledemo.penrose import star

from IPython.core.completerlib import import_re
from conda.history import pretty_diff
# scikit-learn 설치
# !pip install scikit-learn -q
!pip install numpy pandas matplotlib seaborn scikit-learn ipython

In [1]:
import sklearn

sklearn.__version__

'1.9.0'

In [ ]:
# 라이브러리와 한글 폰트 설정
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from IPython.display import display

# 운영체제별 한글 폰트 후보
font_candidates = {
    "Darwin": ["AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"],
    "Windows": ["Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"],
    "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"],
}

available_fonts = {font.name for font in fm.fontManager.ttflist}
selected_font = None

for font_name in font_candidates.get(platform.system(), []):
    if font_name in available_fonts:
        selected_font = font_name
        break

if selected_font:
    plt.rcParams["font.family"] = selected_font
    print("설정된 한글 폰트:", selected_font)
else:
    print("사용 가능한 한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

# 마이너스 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", rc={"axes.unicode_minus": False})
if selected_font:
    sns.set_theme(style="whitegrid", rc={"font.family": selected_font, "axes.unicode_minus": False})

## 생선 이진 분류 - 도미냐? 빙어냐?
<table>
    <tr>
        <td><img src="https://d.pr/i/FaQ4RH+" width="300"></td>
        <td><img src="https://d.pr/i/NmXwPX+" width="300"></td>
    </tr>
</table>


In [ ]:
# 도미와 빙어의 길이/무게 원자료

# 도미 bream 데이터 35개
# bream_length: 도미 35마리의 길이
# bream_weight: 도미 35마리의 무게
bream_length = [25.4, 26.3, 26.5, 29.0, 29.0, 29.7, 29.7, 30.0, 30.0, 30.7, 31.0, 31.0, 31.5, 32.0, 32.0, 32.0, 33.0,
                33.0, 33.5, 33.5, 34.0, 34.0, 34.5, 35.0, 35.0, 35.0, 35.0, 36.0, 36.0, 37.0, 38.5, 38.5, 39.5, 41.0,
                41.0]
bream_weight = [242.0, 290.0, 340.0, 363.0, 430.0, 450.0, 500.0, 390.0, 450.0, 500.0, 475.0, 500.0, 500.0, 340.0, 600.0,
                600.0, 700.0, 700.0, 610.0, 650.0, 575.0, 685.0, 620.0, 680.0, 700.0, 725.0, 720.0, 714.0, 850.0,
                1000.0, 920.0, 955.0, 925.0, 975.0, 950.0]

# 빙어 smelt 데이터 14개
# smelt_length: 빙어 14마리의 길이
# smelt_weight: 빙어 14마리의 무게
smelt_length = [9.8, 10.5, 10.6, 11.0, 11.2, 11.3, 11.8, 11.8, 12.0, 12.2, 12.4, 13.0, 14.3, 15.0]
smelt_weight = [6.7, 7.5, 7.0, 9.7, 9.8, 8.7, 10.0, 9.9, 9.8, 12.2, 13.4, 12.2, 19.7, 19.9]

In [ ]:
# 산점도표를 이용해서 도미와 빙어의 분포를 확인
sns.scatterplot(
    x=bream_length, y=bream_weight, label=' 도미'
)

sns.scatterplot(
    x=smelt_length, y=smelt_weight, label='빙어'
)

plt.legend()
plt.show()

## KNN K-최근접이웃분류모델
데이터포인트간의 거리기반으로 분류를 수행하는 모델


### k-최근접 이웃 알고리즘 작동원리

최근접이웃(K-Nearest Neighbors, KNN)에서 거리를 구하는 방법으로 가장 일반적으로 사용되는 방법은 **유클리드 거리(Euclidean distance)**이다.

유클리드 거리란 "두 점 사이의 직선 거리" 를 구하는 가장 간단한 방법이다.

좌표상 (0, 0)과 (3, 4)라는 점 두 개를 찍었다고 하자. 이 두 점을 선으로 연결하면, 바로 그 선의 길이가 유클리드 거리이다!

$$\text { 거리 }=\sqrt{\left(x_{2}-x_{1}\right)^{2}+\left(y_{2}-y_{1}\right)^{2}}$$

두 점 $p = (p_1, p_2, ..., p_n)$과 $q = (q_1, q_2, ..., q_n)$ 사이의 유클리드 거리는 다음과 같은 수식으로 계산된다.

$$d(p, q) = \sqrt{(p_1 - q_1)^2 + (p_2 - q_2)^2 + \cdots + (p_n - q_n)^2}$$

#### 다양한 거리계산법
이 외에도 거리 계산에 사용하는 방법에는 **맨해튼 거리(Manhattan distance)**, **민코프스키 거리(Minkowski distance)**, **코사인 유사도(Cosine similarity)** 등이 있다.

1. 맨해튼 거리 계산식:

$$d(p, q) = |p_1 - q_1| + |p_2 - q_2| + \cdots + |p_n - q_n|$$

2. 민코프스키 거리 계산식(유클리드와 맨해튼 거리를 일반화한 형태):
  (여기서 $p = 2$일 때는 유클리드 거리, $p = 1$일 때는 맨해튼 거리이다)
$$d(p, q) = \left( \sum_{i=1}^{n} |p_i - q_i|^p \right)^{1/p}$$

3. 코사인 유사도:
  (여기서 $p \cdot q$는 두 벡터의 내적, $\|p\|$와 $\|q\|$는 벡터의 크기이다)
$$\cos \theta = \frac{p \cdot q}{\|p\| \|q\|}$$



In [ ]:
# 도미와 빙어 데이터를 모델 입력용 데이터 X로 변환

# 도미와 빙어의 길이, 무게 list를 이어붙이기
fish_length = bream_length + smelt_length  # 35 + 14 = 49개
fish_weight = bream_weight + smelt_weight

print(len(fish_length), len(fish_weight))

fish_data = [[length, weight] for length, weight in zip(fish_length, fish_weight)]

fish_data = np.array(fish_data)
fish_data


In [ ]:
# fish_data에 대응되는 정답 y 만들기
# 앞에 도미 35마리를 1, 뒤에 빙어 14마리 0으로 지정
fish_target = np.array([1] * 35 + [0] * 14)
fish_target

In [ ]:
# KNeighborsClassifier  - KNN 분류 모델
# KNN은 새 데이터가 입력되면 가장 가까운 학습 데이터 이웃을 찾고,
# 그 이웃들의 정답을 다수결로 사용해 새 데이터의 클래스를 판별함
from sklearn.neighbors import KNeighborsClassifier

# 입력된 값과 가장 가까운 이웃 5개를 보고 판별할 분류 모델 객체 생성
kn_clf = KNeighborsClassifier(n_neighbors=5)

| 항목 | 의미 |
|---|---|
| `n_neighbors = 5` | 가장 가까운 이웃 5개를 보고 판단 |
| `weights = 'uniform'` | 이웃 5개의 투표 가중치를 똑같이 적용 |
| `algorithm = 'auto'` | 가까운 이웃을 찾는 방법은 scikit-learn이 자동 선택 |
| `p = 2` | 거리 계산에서 유클리드 거리 사용 |
| `metric = 'minkowski'` | 민코프스키 거리 사용, `p=2`라서 사실상 유클리드 거리 |
| `n_jobs = None` | 병렬 처리 별도 지정 안 함 |

| 항목 | 의미 |
|---|---|
| `classes_ = [0, 1]` | 이 모델이 배운 정답 종류는 0과 1 |
| `n_features_in_ = 2` | 입력 feature는 2개, 즉 길이와 무게 |
| `n_samples_fit_ = 49` | 학습에 사용한 생선 데이터는 49개 |
| `outputs_2d_ = False` | 정답 `y`가 2차원이 아니라 1차원이라는 뜻 |

In [ ]:
# KNN 분류 모델에 fish_data, fish_target 학습 시키기
# 모델.fit(X, y) - X == 데이터(문제), y == 정답
kn_clf.fit(fish_data, fish_target)

In [ ]:
# score()를 통해 분류 모델의 학습 정확도 점수를 반환(0 ~ 1 우수)

# - 학습에 사용한 데이터를 그대로 평가에 사용 == 당연히 1
kn_clf.score(fish_data, fish_target)

In [ ]:
# 학습된 데이터 말고 새로운 데이터를 전달

# predict() : 학습된 모델로 새 데이터의 정답을 예측하는 메서드
# - 입력은 반드시 2차원의 형태
# sample[0] == 도미 , sample[1] == 빙어
sample = [[40, 500], [10, 10], [11.3, 20], [15, 200]]
kn_clf.predict(sample)  # [1, 0]으로 예측


In [ ]:
# 새 데이터와 가장 가까운 이웃(이미 학습된 데이터) 찾기
# kneighbors(new, n_neighbors=k) : 새 데이터와 가장 가까운 이웃 k개 찾기
# 반환값 distances: 새 데이터와 가까운 이웃 사이의 거리
# 반환값 indices: 가까운 이웃들이 학습 데이터(fish_Data)
#                몇 번째 인덱스에 있는지 나타냄
distances, indices = kn_clf.kneighbors([[20, 170]], n_neighbors=5)
print("distances: ", distances)
print("indices: ", indices)

fish_target[indices]  # 이웃 목록 - 1(도미), 0(빙어)

In [ ]:
# 최근접 이웃을 시각화(산점도)
sns.scatterplot(
    x=fish_data[:, 0],  # 물고기 길이
    y=fish_data[:, 1],  # 물고기 무게
    label='학습 데이터'
)

# 새 데이터(예측하려는 데이터)를 그래프에 표시
sns.scatterplot(x=[40], y=[500], label="새 데이터")

distances, indices = kn_clf.kneighbors([[40, 500]], n_neighbors=5)

# 새 데이터와 가까운 이웃한 데이터의 인덱스를 5개 반환
# print(indices.squeeze())
neighbor_indices = indices.squeeze()

# 가까운 이웃한 데이터를 그래프에 표시
sns.scatterplot(
    x=fish_data[neighbor_indices, 0],  # 이웃 물고기의 길이
    y=fish_data[neighbor_indices, 1],  # 이웃 물고기의 무게
    label="가까운 이웃"
)

plt.xlabel("길이")
plt.ylabel("무게")
plt.show()


## 학습/테스트 세트 분리


In [ ]:
# 모든 데이터를 학습용 사용 안할경우
# 일부 학습, 일부 테스트 할 경우

# train_test_split(): 입력 X와 정답 y를 학습용/테스트용으로 분리함
from sklearn.model_selection import train_test_split

# x는 학습용 데이터(입력값), y는 정답

(X_train, X_test, y_train, y_test) = train_test_split(
    fish_data,  # X(문제)
    fish_target,  # y(답)
    test_size=0.2,  # 전체 데이터 중 20퍼센트를 테스트용, 나머지 80은 학습용
    # 도미와, 빙어의 비율이 달라 비율을 맞춤
    stratify=fish_target,  # 도미,빙어의 train/test 비율을 맞춤
    random_state=42
)

# print("X_train : ", X_train)
# print("X_test : ", X_test)
# print("y_train : ", y_train)
# print("y_test : ", y_test)

print("X_train : ", X_train.shape)
print("X_test : ", X_test.shape)
print("y_train : ", y_train.shape)
print("y_test : ", y_test.shape)


In [ ]:
# 1. 다시 KNN분류 모델 생성 (이웃 모델 찾기)
kn_clf = KNeighborsClassifier()

#2. X_train, y_train만 학습
kn_clf.fit(X_train, y_train)

# 3.평가 : X_test, y_test로 평가 점수 확인
print("테스트 점수:", kn_clf.score(X_test, y_test))

# 4. 예측 : X_test를 모델에 전달할 경우 얻는 예측값을 y_test와 비교
print("예측 : ", kn_clf.predict(X_test))
print("정답 : ", y_test)


## 수상한 도미
도미(길이 25, 무게 150)의 문제


In [ ]:
kn_clf.predict([[25, 100]])
#실행결과 : array([0]) == KNN모델이 0(빙어)으로 예측

In [ ]:
sns.scatterplot(x=X_train[:, 0], y=X_train[:, 1], label='학습 데이터')
sns.scatterplot(x=[25], y=[150], label='수상한 도미')

distances, indices = kn_clf.kneighbors([[25, 150]])
print(distances, indices)

print(y_train[indices])

neighbor_indices = indices.squeeze()
sns.scatterplot(
    x=X_train[neighbor_indices, 0],
    y=X_train[neighbor_indices, 1],
    label='가까운 이웃',
)

plt.xlim((0, 1000))
plt.legend()
plt.show()

## 표준점수-스케일링
서로 다른 속성의 값의 범위를 맞추기 위한 전처리기법.
모델 성능에 직접적인 영향이 있음.


###  표준점수로 환산하기
표준점수(또는 Z-점수)는 데이터가 평균에서 얼마나 떨어져 있는지를 표준편차 단위로 나타낸 값이다.

$Z = \frac{X - \mu}{\sigma}$

-   $X$는 데이터 값
-   $\mu$는 데이터의 평균
-   $\sigma$는 데이터의 표준편차

특성값에서 평균을 빼고, 표준편차로 나누기.


In [ ]:
# preprocessing : 전처리
# StandardScaler : 표준점수
# 사이킷런 제공 스케일링 전처리 클래스
# 스케일리이란? : 데이터의 거리를 공평하도곡 숫자 범위를 맞추는 작업
# -> rkr feature를 z-score로 바꿔주는 전처리기
from sklearn.preprocessing import StandardScaler

sclear = StandardScaler()

# 학습용/테스틍 입력 데이터(X)를 표준점수(z-score) 스케일링
X_train_scaled = sclear.fit_transform(X_train)
X_test_scaled = sclear.transform(X_test)

# 스케일링된 데이터를 fit() 하면 안됨
# 보정데이터 + 기존데이터를 하면 데이터가 엉망진창이 됨 : 데이터의 누수
# -> 대신 학습 데이터를 치환(변환) transform(X:학습데이터) 이용

print(X_test)
print(X_test_scaled)

# -> 데이터가 몰려있는 모습을 확인함

In [ ]:
# 새 KNN분류 모델 만들어서 스케일링된 데이터로 재학습
kn_clf = KNeighborsClassifier()
kn_clf.fit(X_train_scaled, y_train)
print("테스트 점수 : ", kn_clf.score(X_test_scaled, y_test))

In [ ]:
# 수상한 도미 [25,150] 데이터를 스케일링된 데이터를 학습한 모델에게 전달
# -> 수상한 도미 [25,150]를 스케일링하여 전달
q = sclear.transform([[25, 150]])  # 표주점수(z-score)로 치환됨
print(q)

kn_clf.predict(q)  # 스케일링 전==0, 스케일링 후 == 1

In [ ]:
# 스케일링된 데이터를 이용한 시각화
# X_train_scaled[:, 0]: 스케일링된 길이
# X_train_scaled[:, 1]: 스케일링된 무게
sns.scatterplot(x=X_train_scaled[:, 0], y=X_train_scaled[:, 1], label='학습 데이터')

sns.scatterplot(x=q[:, 0], y=q[:, 1], label='수상한 도미')

distances, indices = kn_clf.kneighbors(q)
neighbor_indices = indices.squeeze()

sns.scatterplot(
    x=X_train_scaled[neighbor_indices, 0],
    y=X_train_scaled[neighbor_indices, 1],
    label='가까운 이웃',
)

plt.legend()
plt.show()

# 생선 다중분류

| Fish      | Korean Name | Image | Avg Size |
|-----------|-------------|-------|----------|
| Bream     | 도미        | <img src="https://d.pr/i/FaQ4RH+" alt="Bream" width="300px"> | 평균 길이 50~60cm, 최대 1m |
| Roach     | 붕어        | <img src="https://d.pr/i/C8jz9h+" alt="Roach" width="300px"> | 평균 길이 20~30cm *(일반적인 붕어 기준)* |
| Whitefish | 흰물고기    | <img src="https://d.pr/i/5z0jBB+" alt="Whitefish" width="300px"> | 평균 길이 30~50cm *(종류에 따라 다름)* |
| Parkki    | 파키        | <img src="https://d.pr/i/CcYfbX+" alt="Parkki" width="300px"> | 평균 길이 15~20cm *(일반적인 파키 기준)* |
| Perch     | 농어        | <img src="https://d.pr/i/JvhJwh+" alt="Perch" width="300px"> | 평균 길이 50~60cm, 최대 1m |
| Pike      | 강꼬치고기  | <img src="https://d.pr/i/NNWlsh+" alt="Pike" width="300px"> | 평균 길이 40~55cm, 최대 1m 이상<br>*(북방강꼬치고기 기준)* |
| Smelt     | 빙어        | <img src="https://d.pr/i/NmXwPX+" alt="Smelt" width="300px"> | 평균 길이 10~15cm *(빙어 기준)* |


**참고 및 설명**
- 도미(Bream)는 대표적으로 참돔을 기준으로 하였으며, 평균 길이 50~60cm, 최대 1m까지 자랍니다.
- 농어(Perch)는 실제로는 Perch(배스)와 농어(Seabass)가 다르나, 표 내 농어는 평균 50~60cm, 최대 1m 이상까지 자랍니다.
- 붕어(Roach), 파키(Parkki), 빙어(Smelt) 등은 한국 내 일반적인 평균 크기를 참고하였습니다.
- 흰물고기(Whitefish), 강꼬치고기(Pike)는 여러 종류가 있으나, 대표적인 종의 평균 크기를 기재했습니다.



In [ ]:
# 데이터 불러오기
fish_df = pd.read_csv("./data/fish.csv")
fish_df.head()

In [ ]:
# 데이터셋에 종 별로 몇 마리씩 있는지 확인
fish_df['Species'].value_counts()

In [ ]:
# 다중 분류 모델에 학습시킬 X,y를 입력 X, 정답 y를 분리
# X : 물고기의 수치 컬럼 5개를 담은 2차원 ndarray
# y : 각 물고기의 종류 Species를 담은 1차원 ndarray

X = fish_df[['Weight', 'Length', 'Diagonal', 'Height', 'Weight']].to_numpy()
y = fish_df['Species'].to_numpy()
print(X.shape, y.shape)

In [ ]:
# 다중분류의 train/test split
from sklearn.model_selection import train_test_split

# 전체 데이터 X(문제)와 y(정답)를 학습용 데이터와 테스트용 데이터로 나눔
X_train, X_test, y_train, y_test = train_test_split(
    X,  # 입력 데이터, 예: 생선의 길이와 무게
    y,  # 정답 데이터, 예: 도미/빙어 라벨
    # test_size=0.25,  # 전체 데이터의 20%는 테스트용, 80%는 학습용으로 사용
    stratify=y,  # y의 클래스 비율이 학습용/테스트용에 비슷하게 유지되도록 나눔
    random_state=42  # 실행할 때마다 같은 방식으로 나누기 위한 고정값
)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
# 입격 값 X의 데이터단위를 표준화(z-score) -> 스케일링
from sklearn.preprocessing import StandardScaler

sclaer = StandardScaler()

# 스케일러에서 입력값 X를 학습 -> 내부적으로 펴균, 표준편차가 계산됨
sclaer.fit(X_train)

# 학습 데이터와 test 데이터를 같은 평균/표준편차를 이용해 스케일링
X_train_scaled = sclaer.transform(X_train)
X_test_scaled = sclaer.transform(X_test)

print(X_train_scaled)
print('*' * 30)
print(X_test_scaled)

In [ ]:
# KNN 다중 분류 학습
from sklearn.neighbors import KNeighborsClassifier

# 가까운 이웃 3개를 보고 정답을 예측(다수결)
kn_clf = KNeighborsClassifier(n_neighbors=3)

# 학습
kn_clf.fit(X_train_scaled, y_train)
# kn_clf.get_params()

print(kn_clf.classes_)
print(kn_clf.n_neighbors)
print(kn_clf.n_samples_fit_)

In [ ]:
print("학습 점수 : ", kn_clf.score(X_train_scaled, y_train))
print("테스트 평가 점수 : ", kn_clf.score(X_test_scaled, y_test))

print("예측 : ", kn_clf.predict(X_test_scaled))
print("정답 : ", y_test)


In [ ]:
# KNN이 학습 후 어떤 정보를 가지고 있나 확인
print(kn_clf.classes_)
print(kn_clf._fit_X)  # 학습 입력 데이터 X
print(kn_clf._y)  # 학습 정답 데이터 y

In [ ]:
print(X_test_scaled.shape)

In [ ]:
print(X_test_scaled[:5])  # X_test_scaled 0~4번행
# predict :예측
# 테스트 스케일의 5행가지 예측하고 정답과 비교
print(kn_clf.predict(X_test_scaled[:5]))
print(y_test[:5])

In [56]:
# 예측 확률 확인
kn_clf.predict_proba(X_test_scaled[:5])
# print( np.round(kn_clf.predict_proba(X_test_scaled[:5]),2) )

array([[0.        , 0.        , 0.66666667, 0.        , 0.33333333,
        0.        , 0.        ],
       [0.        , 0.        , 0.66666667, 0.        , 0.33333333,
        0.        , 0.        ],
       [0.        , 0.        , 0.66666667, 0.        , 0.33333333,
        0.        , 0.        ],
       [0.        , 1.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.66666667, 0.        , 0.        , 0.33333333,
        0.        , 0.        ]])

In [66]:
# 다중 분류에서 kneighbor() 예측 근거가 된 이웃찾기
distances, indices = kn_clf.kneighbors(X_test_scaled[:1])
print(X_test_scaled[0])
print(distances)
print(indices)

print(y_train[indices])

[-0.84520127 -0.75807703 -0.71391456 -0.70656438 -0.84520127]
[[0.0723414  0.0990876  0.10758785]]
[[100 117  69]]
[['Roach' 'Perch' 'Perch']]


## @실습문제: 붓꽃 다중 분류

붓꽃은 세포핵을 둘러싼 꽃받침(sepal)과 꽃잎(petal)으로 구성되어 있으며, 이러한 특징을 기반으로 붓꽃의 품종을 식별할 수 있습니다.

- Setosa (세토사): Setosa는 붓꽃 중에서 가장 작은 꽃잎과 꽃받침을 가지고 있습니다.
꽃잎과 꽃받침이 비교적 짧고 뾰족한 모습을 갖고 있으며, 주로 흰색 또는 연한 분홍색을 띠고 있습니다.

- Versicolor (버시컬러): Versicolor는 Setosa보다 크고 긴 꽃잎과 꽃받침을 가지고 있습니다.
꽃잎의 색은 보통 연한 보라색이며, 중간 크기의 붓꽃입니다.

- Virginica (버지니카): Virginica는 붓꽃 중에서 가장 크고 긴 꽃잎과 꽃받침을 가지고 있습니다.
꽃잎의 색은 주로 짙은 보라색이며, 다른 품종들에 비해 상대적으로 더 큰 크기를 갖고 있습니다.

![](https://d.pr/i/4egoon+)


In [79]:
# 사이킷런에서 아이리스 데이터셋 얻어오기
from sklearn.datasets import load_iris

iris = load_iris()
# print(iris.data) # 입력값 X
# print(iris.target) # 정답 y

print(iris.feature_names)
print(iris.target_names)

iris_df = pd.DataFrame(
    data=iris.data,  # X를 지정함 == 입력값을 지정함
    columns=iris.feature_names)
# display(iris_df.head())

iris_df['Species'] = iris.target

iris_df

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
['setosa' 'versicolor' 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),Species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [85]:
  # train = 공부용 문제
# test = 시험용 문제
# 컴퓨터에게 모든 문제와 정답을 다 보여주면,
# 진짜로 배운 건지 그냥 외운 건지 알 수 없음
# 그래서 일부는 공부용(train), 일부는 시험용(test)으로 따로 나눔

X_train, X_test, y_train, y_test = train_test_split(
    iris.data,  # X: 문제, 예: 꽃잎 길이, 꽃잎 너비 같은 숫자 정보
    iris.target,  # y: 정답, 예: 꽃의 종류
    stratify=iris.target,  # 꽃 종류 비율이 train/test에 비슷하게 들어가게 나눔
    random_state=42  # 실행할 때마다 똑같이 나누기 위한 숫자
)

# X_train: 공부할 문제
# y_train: 공부할 문제의 정답
# X_test: 시험 볼 문제
# y_test: 시험 볼 문제의 정답

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# 2. 스케일링: 숫자 크기를 비슷하게 맞추는 작업
# 예를 들어 어떤 값은 1~10이고, 어떤 값은 100~1000이면
# 큰 숫자가 더 중요해 보일 수 있음
# 그래서 공평하게 비교하도록 숫자 범위를 맞춤

scaler = StandardScaler()

# fit: X_train의 평균과 표준편차를 계산해서 scaler 안에 저장함
# 시험 문제인 X_test를 기준으로 fit하면 안 됨
# 시험지는 공부할 때 미리 보면 안 되기 때문
scaler.fit(X_train)

# transform: 저장해 둔 평균과 표준편차로 숫자 크기를 바꿈
X_train_scaled = scaler.transform(X_train)  # 공부용 문제를 스케일링
X_test_scaled = scaler.transform(X_test)  # 시험용 문제도 같은 기준으로 스케일링

# 3. KNN 분류 모델 생성 + 학습
# KNN은 새 문제가 들어오면 가까운 친구들을 보고 정답을 맞힘
# n_neighbors=3: 가장 가까운 친구 3명을 보고 다수결로 정답을 정함

kn_clf = KNeighborsClassifier(n_neighbors=3)

# fit: 공부용 문제와 정답을 보고 학습함
kn_clf.fit(X_train_scaled, y_train)

# 4. Score 평가
# score는 모델이 정답을 얼마나 잘 맞혔는지 점수로 보여줌

# 공부했던 문제를 다시 풀게 한 점수
print("학습 데이터셋 평가 :", kn_clf.score(X_train_scaled, y_train))

# 처음 보는 시험 문제를 풀게 한 점수
# 이 점수가 더 중요함
print("테스트 데이터셋 평가 :", kn_clf.score(X_test_scaled, y_test))


# 5. 예측 predict()
# 시험 문제 중 앞에서 5개만 꺼내서 모델에게 맞혀보라고 함

# 모델이 생각한 정답
print("예측정답: ", kn_clf.predict(X_test_scaled[:5]))

# 진짜 정답
print("실제정답: ", y_test[:5])

(112, 4) (112,)
(38, 4) (38,)
학습 데이터셋 평가 : 0.9210526315789473
테스트 데이터셋 평가 : 0.9210526315789473
예측정답:  [0 1 1 1 0]
실제정답:  [0 1 1 1 0]
